# rustymath

> Maths in plain Rust, from fast f16, f32 and f64 functions to exact rationals.

rustymath gives you maths in plain Rust, and in Python with NumPy, from fast functions on 16-, 32- and 64-bit floats to exact rationals. It covers the elementary functions, such as `exp`, `log`, `sin` and `pow`, at 16, 32 and 64 bits. Most of them also take complex numbers. It also has special functions such as `gamma` and `erf`, probability distributions, reductions and scans, searching and sorting, linear algebra, polynomials, number theory, and APL-inspired tolerant comparison. In both languages, it also computes exactly on rationals: matrix solves and determinants, polynomials, gcds and binomial coefficients.

In Rust, you call each function on single values inside your own loops, or through an object that runs it over a whole slice. In Python, each maths function is a NumPy ufunc. The linear algebra functions run as generalized ufuncs, over stacks of matrices.

Its float functions give up the last bits of accuracy for speed. Every elementary and special function is within 4 units in the last place (ulps) of the correctly rounded result. At 32 bits, most functions run two to six times as fast as the standard library's versions.

## Install

Install the Python package from PyPI. For Rust, see [Using rustymath from Rust](#using-rustymath-from-rust).

```sh
pip install rustymath
```

## Using rustymath from Python

The maths functions are NumPy ufuncs. Each takes arrays of any shape, broadcasts them as NumPy does, and gives a result of the same width. A complex array works wherever the function has complex values. An integer array works wherever NumPy's version takes one. `add`, `multiply`, `maximum` and `minimum` have `reduce` and `accumulate` methods that run rustymath's vectorized sums, products, maximums and minimums. Import rustymath beside NumPy:

In [ ]:
import numpy as np, rustymath as rm

Where NumPy has a function, rustymath uses the same name and arguments. To switch, you change only the import. Here `exp` keeps the 32-bit width of its argument:

In [ ]:
x = np.linspace(0, 2, 5, dtype=np.float32)
rm.exp(x)

array([1.       , 1.6487211, 2.718282 , 4.4816885, 7.389056 ],
      dtype=float32)

To check the accuracy yourself, compare with NumPy. `np.spacing` gives the size of one ulp at each value. Dividing the difference by it gives the distance between the two results in ulps:

In [ ]:
xs = np.linspace(-10, 10, 100_001)
y = np.exp(xs)
(np.abs(rm.exp(xs) - y) / np.spacing(y)).max()

np.float64(2.0)

Functions that NumPy lacks keep their own names. `root(n, y)` gives the `n`th root of `y`:

In [ ]:
rm.root(3, np.array([8., 27., 1000.]))

array([ 2.,  3., 10.])

The Rust names follow Rust's standard library instead. Python's `log`, `power`, `arcsin` and `arctan2` are Rust's `ln`, `pow`, `asin` and `atan2`.

### Complex numbers

A complex array gives complex results, at each function's principal value. For `exp`, `log`, `power`, `sin`, `cos`, `sinh` and `cosh`, rustymath computes a complex array through its vectorized real loops:

In [ ]:
rm.sqrt(np.array([-4+0j, 3+4j])), rm.log(np.array([-1+0j]))

(array([0.+2.j, 2.+1.j]), array([0.+3.14159265j]))

### Special functions and distributions

The special functions take SciPy's names, such as `gamma`, `gammaln`, `erf`, `erfinv`, `betainc` and `lambertw`:

In [ ]:
rm.gamma(np.array([0.5, 5, 10])), rm.erfinv(0.5)

(array([1.77245385e+00, 2.40000000e+01, 3.62880000e+05]),
 np.float64(0.4769362762044699))

`rm.distribution` holds sixteen distributions. Each constructor takes the distribution's parameters, or uses standard ones where the distribution has them. The result has `pdf`, `cdf`, `quantile` and `sample`. The same `seed` gives the same draws:

In [ ]:
n = rm.distribution.normal()
n.cdf([-1.96, 0, 1.96]), n.quantile(0.975), n.sample(3, seed=0)

(array([0.0249979, 0.5      , 0.9750021]),
 1.9599639845400538,
 array([ 0.11623959, -0.78285485, -2.33463531]))

### Polynomials

`rm.polynomial` evaluates polynomials, finds roots, constructs coefficients from roots, differentiates and fits data, under the names of `numpy.polynomial.polynomial`. Coefficients start from the constant term. Evaluation, construction and derivatives take real or complex arrays. `polyvalfromroots(x, roots, m=1)` evaluates directly from roots without forming coefficients. The roots of x^3 - 6x^2 + 11x - 6 are 1, 2 and 3:

In [ ]:
rm.polynomial.polyroots([-6, 11, -6, 1])

array([1.+0.j, 2.-0.j, 3.+0.j])

### Linear algebra

`rm.linalg` provides real and complex matrix solves and decompositions, including `solve`, `det`, `lu`, `cholesky`, `qr`, `svd`, `eigh`, `eig`, `lstsq`, `pinv` and `matrix_rank`. They take stacks of matrices and broadcast the leading axes. `solve` and `pinv` also handle singular or non-square matrices through least-norm least-squares solutions. `permanent` takes square matrices; `echelon` reduces one 2-D matrix. LU factors store packed data and row positions, with `.l` and `.u` properties for the triangular arrays. Reuse the factors for solves, inverses and determinants:

In [ ]:
a = np.array([[4., 1], [1, 3]])
f = rm.linalg.lu(a)
f.solve([1., 2]), f.determinant(), rm.linalg.eigh(a).values, rm.linalg.solve([[1., 2], [2, 4]], [1., 2])

(array([0.09090909, 0.63636364]),
 np.float64(11.0),
 array([2.38196601, 4.61803399]),
 array([0.2, 0.4]))

`rm.matmul` multiplies matrices as `np.matmul` does, with integer matrices in int64. `rm.linalg.semiring(f, g)` gives the matrix product whose sums use `f` and whose products use `g`, as APL's inner product `f.g` does. With `add` and `equal`, it counts the matching items. Squaring a matrix of edge lengths with `minimum` and `add` gives the shortest paths of at most two edges:

In [ ]:
d = np.array([[0, 3, np.inf], [np.inf, 0, 1], [2, np.inf, 0]])
minplus = rm.linalg.semiring(rm.minimum, rm.add)
minplus(d, d)

array([[0., 3., 4.],
       [3., 0., 1.],
       [2., 5., 0.]])

### Number theory

`rm.number_theory` tests primality, factorises, finds the next or previous prime, gives Euler's totient and exact factorials, and lists and counts primes, on Python integers of any size. Ctrl-C stops a long call:

In [ ]:
nt = rm.number_theory
nt.factor(2**64 + 1), nt.prime_count(10**8), nt.next_prime(10**20)

([(274177, 1), (67280421310721, 1)], 5761455, 100000000000000000039)

### Exact arithmetic

`rm.exact` takes integers and `fractions.Fraction`. It provides rational gcd, lcm, remainder and binomial coefficients, matrix solves, pseudoinverses, determinants, permanents and row reduction, and polynomial construction, evaluation and derivatives. Elementwise calls broadcast and return Fractions or NumPy object arrays. Matrix calls take one 2-D matrix. Ordinary `rm.linalg` calls stay floating-point.

In [ ]:
from fractions import Fraction

In [ ]:
rm.exact.solve([[2, 1], [1, 2]], [Fraction(1, 2), 0]), rm.exact.binom(Fraction(3, 2), 2)

(array([Fraction(1, 3), Fraction(-1, 6)], dtype=object), Fraction(3, 8))

### Tolerant comparison

APL compares floats within a relative tolerance. Two floats are equal when their difference is less than the tolerance times the larger magnitude. With the usual tolerance, `0.1 + 0.2` equals `0.3`. `rm.equal`, `not_equal`, `less`, `less_equal`, `greater` and `greater_equal` compare this way. Their tolerance `ct` defaults to `rm.CT`, 1e-14. Pass `ct=0` to compare exactly:

In [ ]:
rm.equal(0.1 + 0.2, 0.3), rm.equal(0.1 + 0.2, 0.3, ct=0)

(np.True_, np.False_)

`floor`, `ceil`, `remainder`, `gcd` and `lcm` take `ct` too. A value within tolerance of a whole number counts as that whole number:

In [ ]:
rm.floor(np.array([2.9999999999999996, 2.5]))

array([3., 2.])

`remainder` takes its arguments in NumPy's order. `rm.remainder(x, y)` is the Rust `residue(y, x)`.

`unique`, `index` and `isin` search floats within the same tolerance. They match Booleans and integers exactly. `unique` gives the values in order of first occurrence, with each one's first position, each item's class number and each class's size. `index(x, y)` gives where in `x` each item of `y` first matches, or the size of `x` where nothing does:

In [ ]:
rm.unique([0.3, 1.0, 0.1 + 0.2]), rm.index([0.3, 1.0], 0.1 + 0.2)

(Unique(values=array([0.3, 1. ]), firsts=array([0, 1]), numbers=array([0, 1, 0]), counts=array([2, 1])),
 np.int64(0))

`rm.search` has stable `argsort` and `lexsort`, `searchsorted`, integer `bincount`, class `counts` and `group_indices`. These functions are also available directly on `rm`. Sorting and insertion searches compare exactly and put NaNs last. `rm.integer.encode` and `decode` convert mixed-radix integers with checked int64 arithmetic; `rm.polynomial.decode` and `rm.exact.decode` accept field-valued bases and digits.

`rm.minmax(x, axis=-1)` computes both bounds in one pass. Boolean `logical_and`, `logical_or`, `logical_xor` and `logical_xnor` have reduction and accumulation methods. `rm.logical_scan(x, table, axis=-1)` scans any Boolean dyad given by its 2×2 truth table. `rm.set_threads(n)` sets the calling thread's limit and returns its previous setting. Use 0 for the default or 1 for serial calls.

{{< include ../src/lib.md >}}

## Licence

Apache-2.0. Some algorithms and constants come from SLEEF and from Arm Optimized Routines. `THIRD-PARTY.md` has their notices.